[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/08_knn/02_from_scratch.ipynb)

# Bài 02 — Cài đặt KNN from scratch

**🎯 Mục tiêu**
- Cài đặt KNN classifier & regressor: từ phiên bản vòng lặp dễ hiểu → phiên bản **vector hoá** nhanh gấp hàng chục – hàng trăm lần.
- Xử lý **hoà phiếu**, **KNN có trọng số theo khoảng cách**.
- Hiểu độ phức tạp tính toán và các cấu trúc dữ liệu tăng tốc (KD-tree, Ball tree, approximate NN).
- Đối chiếu với scikit-learn.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/08_knn

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from ml_course.datasets import load_digits_data
from ml_course.visualization import set_style

set_style()
X, y, images = load_digits_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)

## 1. Phiên bản 1 — vòng lặp, đúng theo định nghĩa

In [ ]:
def knn_predict_one(x, X_train, y_train, k):
    distances = [np.sqrt(np.sum((x - xi) ** 2)) for xi in X_train]     # 1. khoảng cách tới mọi điểm
    nearest = np.argsort(distances)[:k]                                  # 2. k điểm gần nhất
    return Counter(y_train[nearest]).most_common(1)[0][0]                # 3. bỏ phiếu


start = time.perf_counter()
pred_loop = np.array([knn_predict_one(x, X_train, y_train, k=5) for x in X_test])
t_loop = time.perf_counter() - start
print(f"Accuracy = {np.mean(pred_loop == y_test):.4f} | thời gian: {t_loop:.2f}s")

## 2. Phiên bản 2 — vector hoá

Tính **ma trận khoảng cách** $(n_{test} \times n_{train})$ bằng một phép nhân ma trận (chương 01, bài 03):

$$\lVert\mathbf a - \mathbf b\rVert^2 = \lVert\mathbf a\rVert^2 + \lVert\mathbf b\rVert^2 - 2\,\mathbf a^\top\mathbf b$$

Dùng `np.argpartition` (chỉ cần $k$ phần tử nhỏ nhất, không cần sắp xếp toàn bộ — $O(n)$ thay vì $O(n\log n)$):

In [ ]:
class KNNClassifier:
    def __init__(self, k=5, weights="uniform"):
        self.k, self.weights = k, weights

    def fit(self, X, y):
        self.X_, self.y_ = np.asarray(X, float), np.asarray(y)
        self.classes_ = np.unique(self.y_)
        return self                                  # "train" = ghi nhớ dữ liệu

    def _distances(self, X):
        sq = (X**2).sum(1)[:, None] + (self.X_**2).sum(1)[None, :] - 2 * X @ self.X_.T
        return np.sqrt(np.maximum(sq, 0))

    def predict_proba(self, X):
        D = self._distances(np.asarray(X, float))
        idx = np.argpartition(D, self.k - 1, axis=1)[:, : self.k]      # k láng giềng (chưa sắp xếp)
        d = np.take_along_axis(D, idx, axis=1)
        w = 1 / np.maximum(d, 1e-12) if self.weights == "distance" else np.ones_like(d)
        labels = self.y_[idx]
        votes = np.stack([(w * (labels == c)).sum(1) for c in self.classes_], axis=1)
        return votes / votes.sum(1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


start = time.perf_counter()
pred_vec = KNNClassifier(k=5).fit(X_train, y_train).predict(X_test)
t_vec = time.perf_counter() - start
print(f"Accuracy = {np.mean(pred_vec == y_test):.4f} | thời gian: {t_vec * 1000:.1f} ms | nhanh hơn ~{t_loop / t_vec:.0f} lần")
print("Hai phiên bản cho cùng dự đoán:", np.mean(pred_vec == pred_loop))

## 3. Hoà phiếu và KNN có trọng số

**Hoà phiếu** (vd $k = 4$: 2–2): cách xử lý phổ biến — chọn lớp có láng giềng **gần nhất**, hoặc chọn $k$ lẻ cho bài toán 2 lớp. Trong code trên, `np.argmax` chọn lớp có chỉ số nhỏ hơn khi hoà — đơn giản nhưng hơi tuỳ tiện.

**KNN có trọng số khoảng cách:** láng giềng càng gần càng có tiếng nói lớn, $w_i = 1/d_i$. Giảm phụ thuộc vào việc chọn $k$ và giải quyết phần lớn trường hợp hoà.

In [ ]:
for k in [1, 3, 10, 30]:
    acc_u = np.mean(KNNClassifier(k=k).fit(X_train, y_train).predict(X_test) == y_test)
    acc_d = np.mean(KNNClassifier(k=k, weights="distance").fit(X_train, y_train).predict(X_test) == y_test)
    print(f"k = {k:2d}: uniform = {acc_u:.4f} | distance-weighted = {acc_d:.4f}")

## 4. KNN regressor

In [ ]:
class KNNRegressor(KNNClassifier):
    def predict(self, X):
        D = self._distances(np.asarray(X, float))
        idx = np.argpartition(D, self.k - 1, axis=1)[:, : self.k]
        return self.y_[idx].astype(float).mean(axis=1)


from sklearn.neighbors import KNeighborsRegressor

from ml_course.datasets import make_regression_1d

Xr, yr = make_regression_1d(n=200, kind="sin", noise=0.4, seed=1)
ours = KNNRegressor(k=7).fit(Xr, yr).predict(Xr[:5])
ref = KNeighborsRegressor(7).fit(Xr, yr).predict(Xr[:5])
print("scratch:", ours.round(4), "\nsklearn:", ref.round(4))

Dự đoán của KNN hồi quy là **trung bình** của $k$ láng giềng → một hàm bậc thang theo $x$. Số láng giềng điều khiển đánh đổi bias–variance:

In [ ]:
grid = np.linspace(Xr.min(), Xr.max(), 500).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(Xr[:, 0], yr, s=12, color="0.6", label="dữ liệu train")
for k, color in [(1, "C3"), (7, "C0"), (60, "C2")]:
    ax.plot(grid[:, 0], KNNRegressor(k=k).fit(Xr, yr).predict(grid), color=color, lw=2, label=f"k = {k}")
ax.legend()
ax.set_title("KNN hồi quy: k = 1 bám từng điểm nhiễu, k quá lớn làm phẳng cả quy luật")
plt.show()

## 5. So sánh với scikit-learn và phiên bản tham chiếu

In [ ]:
from ml_course.algorithms import KNNClassifierScratch

sk = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
ref = KNNClassifierScratch(k=5).fit(X_train, y_train)
print(f"sklearn          : {sk.score(X_test, y_test):.4f}")
print(f"KNNClassifierScratch (src/ml_course): {np.mean(ref.predict(X_test) == y_test):.4f}")
print(f"Bài này          : {np.mean(pred_vec == y_test):.4f}")

## 6. Độ phức tạp và cách tăng tốc

| | Brute force | KD-tree / Ball tree |
|---|---|---|
| "Train" | $O(1)$ (lưu dữ liệu) | $O(n\log n)$ dựng cây |
| Dự đoán 1 điểm | $O(n\,d)$ | ~$O(\log n)$ khi $d$ rất nhỏ (vài chiều); suy giảm về (thậm chí chậm hơn) brute force khi $d$ tăng |
| Bộ nhớ | $O(n\,d)$ — phải giữ **toàn bộ** dữ liệu train | như trên + cây |

- scikit-learn tự chọn (`algorithm="auto"`): KD-tree/Ball tree cho ít chiều, brute force cho nhiều chiều.
- Với hàng triệu – tỉ vector nhiều chiều (vd: tìm kiếm ảnh, embedding văn bản): **approximate nearest neighbor** (FAISS, Annoy, HNSW) — chấp nhận sai số nhỏ để nhanh hơn hàng nghìn lần (chương 32 — retrieval).

In [ ]:
from sklearn.datasets import make_blobs

for d in [2, 10, 50]:
    Xb, _ = make_blobs(20000, n_features=d, random_state=0)
    q = Xb[:500]
    for algo in ["brute", "kd_tree"]:
        nn = KNeighborsClassifier(5, algorithm=algo).fit(Xb, np.zeros(len(Xb)))
        start = time.perf_counter()
        nn.kneighbors(q)
        print(f"d = {d:2d} | {algo:7s}: {(time.perf_counter() - start) * 1000:7.1f} ms")

KD-tree rất nhanh ở 2 chiều nhưng mất lợi thế khi số chiều tăng — trên máy chạy notebook này nó đã **chậm hơn** brute force từ $d = 10$ (brute force được tăng tốc bằng phép nhân ma trận BLAS). Lại là **curse of dimensionality** (chương 05, bài 06): trong nhiều chiều, cây không loại được nhánh nào nên phải duyệt gần hết.

## 🧠 Kiểm tra nhanh

1. Vì sao dùng `argpartition` thay vì `argsort` để tìm $k$ láng giềng?
2. Dự đoán cho 10 000 điểm với tập train 1 triệu điểm, 100 chiều, brute force: khoảng bao nhiêu phép tính?
3. KNN có trọng số khoảng cách xử lý trường hợp một điểm test trùng đúng một điểm train thế nào?

## 📌 Tóm tắt

- KNN = ma trận khoảng cách + chọn $k$ nhỏ nhất + bỏ phiếu/trung bình.
- Vector hoá (nhân ma trận) nhanh hơn vòng lặp hàng chục – hàng trăm lần (tuỳ kích thước dữ liệu).
- Chi phí nằm ở **lúc dự đoán**: $O(nd)$ mỗi điểm; KD-tree chỉ giúp khi ít chiều; ANN cho quy mô lớn.